# Marketing Campaign Pipeline Notebook

In [ ]:
# 📌 Cell 1 — Import all libraries
# RandomForestRegressor is added here for the revenue prediction model

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing   import LabelEncoder, MultiLabelBinarizer
from sklearn.ensemble        import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics         import accuracy_score, f1_score
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score

print("MARKETING CAMPAIGN")

In [ ]:
# 📌 Cell 2 — Load all 3 brand CSV files and merge into one DataFrame
# We add Campaign_Name before merging so we know which row belongs to which brand

print("Step 1 — Loading 3 datasets")

nykaa   = pd.read_csv('nykaa_campaign_data_with_nulls.csv')
purplle = pd.read_csv('purplle_campaign_data_with_nulls.csv')
tira    = pd.read_csv('tira_campaign_data_with_nulls.csv')

nykaa['Campaign_Name']   = 'Nykaa'
purplle['Campaign_Name'] = 'Purplle'
tira['Campaign_Name']    = 'Tira'

df = pd.concat([nykaa, purplle, tira], ignore_index=True)
df.drop(columns=['Campaign_ID'], inplace=True)
print(f"Merged shape : {df.shape}")

In [ ]:
# 📌 Cell 3 — Handle missing values
# Categorical → mode (most frequent), Numerical → median (safe against outliers)

print("Step 2 — Cleaning null values")

cat_cols = ['Campaign_Type', 'Target_Audience', 'Channel_Used',
            'Language', 'Customer_Segment', 'Campaign_Name']
num_cols = ['Duration', 'Impressions', 'Clicks', 'Leads',
            'Conversions', 'Revenue', 'Acquisition_Cost',
            'ROI', 'Engagement_Score']

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

df['Date'] = df['Date'].ffill()
df[['Duration','Impressions','Clicks','Leads','Conversions']] = \
    df[['Duration','Impressions','Clicks','Leads','Conversions']].astype(int)

print(f"Nulls remaining : {df.isnull().sum().sum()}")

In [ ]:
# 📌 Cell 4 — Feature engineering
# 1. Create Profit_Flag target column from ROI
# 2. Multi-label encode Channel_Used into 6 binary columns
# 3. Label encode all remaining categorical columns

print("Step 3 — Feature engineering")

# Target column: 1 = Profit, 0 = Loss
df['Profit_Flag'] = (df['ROI'] > 0).astype(int)

# Channel_Used → 6 binary columns (Email, Facebook, Google, Instagram, WhatsApp, YouTube)
mlb      = MultiLabelBinarizer()
ch_lists = df['Channel_Used'].apply(
    lambda x: [c.strip() for c in str(x).split(',')]
)
ch_df = pd.DataFrame(
    mlb.fit_transform(ch_lists),
    columns=mlb.classes_, index=df.index
)
df = pd.concat([df, ch_df], axis=1)
df.drop(columns=['Channel_Used', 'Date'], inplace=True)

# Encode text columns → numbers
le = LabelEncoder()
for col in ['Campaign_Name','Campaign_Type','Target_Audience',
            'Language','Customer_Segment']:
    df[col] = le.fit_transform(df[col].astype(str))

df.to_csv('clean_campaign_data.csv', index=False)
print(f"clean_campaign_data.csv saved  {df.shape}")

In [ ]:
# 📌 Cell 5 — Basic EDA summary
# Quick sanity check on the data before training

print("Step 4 — Basic EDA summary")
print(f"Total campaigns : {len(df):,}")
print(f"Profit (1)      : {(df['Profit_Flag']==1).sum():,}  ({df['Profit_Flag'].mean()*100:.1f}%)")
print(f"Loss   (0)      : {(df['Profit_Flag']==0).sum():,}  ({(1-df['Profit_Flag'].mean())*100:.1f}%)")
print(f"Avg Revenue     : ₹{df['Revenue'].mean():,.0f}")
print(f"Avg ROI         : {df['ROI'].mean():.3f}")
print(f"Avg Engagement  : {df['Engagement_Score'].mean():.2f}")

In [ ]:
# 📌 Cell 6 — Train Revenue Prediction Model (Regression)
#
# Goal       : Predict how much Revenue a campaign will generate (a number, not a class)
# Model type : RandomForestRegressor  ← regression, not classification
# Features   : 17  (same as the classifier MINUS Revenue)
# Target     : Revenue
#
# IMPORTANT: Revenue is the TARGET here — we cannot use it as an input feature.
# That is why this model has 17 features while the classifier has 18.
#
# Evaluation metrics for regression (NOT accuracy):
#   RMSE → average prediction error in ₹ (lower is better)
#   MAE  → average absolute error in ₹  (lower is better)
#   R²   → how much revenue variance the model explains (1.0 = perfect)

print("Step 5 — Training revenue prediction model (regression)...")

REV_FEATURES = [
    'Campaign_Name', 'Campaign_Type', 'Target_Audience', 'Language',
    'Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions',
    'Acquisition_Cost', 'Engagement_Score',
    'Email', 'Facebook', 'Google', 'Instagram', 'WhatsApp', 'YouTube'
]

X_rev = df[REV_FEATURES]
y_rev = df['Revenue']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_rev, y_rev, test_size=0.2, random_state=42
)

rev_model = RandomForestRegressor(n_estimators=80, random_state=42, n_jobs=-1)
rev_model.fit(X_train_r, y_train_r)

y_pred_r = rev_model.predict(X_test_r)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
mae  = mean_absolute_error(y_test_r, y_pred_r)
r2   = r2_score(y_test_r, y_pred_r)

print(f"    RMSE : ₹{rmse:,.0f}")
print(f"    MAE  : ₹{mae:,.0f}")
print(f"    R²   : {r2:.4f}")

with open('revenue_model.pkl', 'wb') as f:
    pickle.dump(rev_model, f)

print("revenue_model.pkl saved!")

In [ ]:
# 📌 Cell 7 — Train Profit / Loss Classification Model
#
# Goal       : Predict if a campaign will be Profit (1) or Loss (0)
# Model type : RandomForestClassifier
# Features   : 18  (Revenue IS included — it helps predict profitability)
# Target     : Profit_Flag  (1 if ROI > 0, else 0)
#
# NOTE: ROI is NOT a feature — Profit_Flag is derived from ROI,
# so including ROI would be data leakage.

print("Step 6 — Training classification model...")

FEATURES = [
    'Campaign_Name', 'Campaign_Type', 'Target_Audience', 'Language',
    'Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions',
    'Revenue', 'Acquisition_Cost', 'Engagement_Score',
    'Email', 'Facebook', 'Google', 'Instagram', 'WhatsApp', 'YouTube'
]

X = df[FEATURES]
y = df['Profit_Flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=80, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"    Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"    F1 Score : {f1_score(y_test, y_pred)*100:.2f}%")

In [ ]:
# 📌 Cell 8 — Save the classification model

with open('profit_loss_classifier.pkl', 'wb') as f:
    pickle.dump(model, f)

print("profit_loss_classifier.pkl saved!")